# Conditional flow matching: endpoint residual

Architecture: `Linear(2, hidden_size) → ReLU → Linear(hidden_size, 1)` with input `[x, y_t]`. The network predicts the **endpoint residual** `r = y_1-y_t`. Reconstruct the clean endpoint as `endpoint_hat = y_t + residual_hat`. All `4*hidden_size+1` parameters are trained jointly. `hidden_size` sets the number of hidden neurons; `total_steps` sets the number of updates.

The path from Gaussian noise `z` to the target is a straight line:

$$y_t=(1-t)z+t y_1,\qquad z\sim\mathcal N(0,1).$$

**Training:** `train_sampling_steps=None` draws continuous `t ~ Uniform(0, 1)`. An integer `train_sampling_steps=T` draws `t` from `0, 1/T, ..., (T-1)/T`. Construct `y_t` and pass `[x,y_t]` to the network. The loss is **MSE(residual_hat, y_1-y_t)**, with one forward/backward pass per update. This residual loss equals `(1-t)^2` times the corresponding velocity MSE and also equals the reconstructed endpoint MSE.

**Sampling:** run `eval_sampling_steps` Euler steps with the same network, independently of the training grid. Compute velocity from the predicted residual:

$$v_\theta(x,y_t,t)=\frac{\widehat r(x,y_t)}{1-t},\qquad y_{k+1}=y_k+\Delta t\,v_\theta(x,y_k,t_k).$$

The final network call is at `t=(T-1)/T`, avoiding division by zero; the final update produces `y_t+residual_hat`. The sampler never receives the target. The MLP has exactly two inputs; time is used only by the schedule. For deterministic `y_1=f(x)`, the residual is `f(x)-y_t` without a separate time input.

The upper chart shows MSE after full Euler sampling; the lower chart shows the fit and target function. Evaluation and extrapolation start from zero for every x; Gaussian noise remains random during training. Only the renderer performs evaluation, without a frame queue. Extrapolation is shared in `_shared/extrapolation.py`.

Original method and linear paths: [Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747).

In [1]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import torch
from _shared import Config
from _shared.conditional_flow_matching import FlowMatchingConfig
from _shared.flow_matching_experiment import FlowMatchingExperiment
from _shared.visualization import train_live

torch.set_num_threads(1)

In [2]:
config = Config(
    seed=42,
    hidden_size=32,
    total_steps=20_000,
    learning_rate=0.01,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
flow_config = FlowMatchingConfig(
    eval_sampling_steps=1,
    train_sampling_steps=None,
)
target_fps = 10

In [3]:
flow_experiment = FlowMatchingExperiment(config, flow_config)
await train_live(flow_experiment, target_fps=target_fps)

In [4]:
from _shared.extrapolation import show_extrapolation

show_extrapolation(flow_experiment, padding_fraction=0.10)